<a href="https://colab.research.google.com/github/databyhuseyn/DeepLearning/blob/main/Natural_Langauge_Processing_with_RNNs_and_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.9 MB/s eta 0:00:00


In [2]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [3]:
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [4]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

When running a Jupyter/Colab notebook, the output of each cell gets saved in the Out dictionary, so if the output of a cell is a large model or tensor, then it's not enough to delete the variable, you must also delete the output from the Out dictionary (e.g., by clearing the whole dictionary with Out.clear()).

# Generating Shakespearean Text Using a Character Level RNN

In [5]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [6]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [7]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [8]:
stoi = {char: index for index, char in enumerate(vocab)}
itos = {index: char for index, char in enumerate(vocab)}

In [9]:
stoi["a"]

13

In [10]:
itos[13]

'a'

In [11]:
import torch

def encode_text(text):
    return torch.tensor([stoi[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([itos[char_id.item()] for char_id in char_ids])

# char_id  tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])
# tensor([27]).item() --> 27

In [12]:
encoded = encode_text("Hello, world!")
encoded

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [13]:
decode_text(encoded)

'hello, world!'

In [14]:
# salamlar

# Conv2d / Conv1d

# sala --> alam
# alam --> laml
# laml --> amla
# amla --> mlar

In [15]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx : end]
        target = self.encoded_text[idx + 1 : end + 1]
        return window, target

In [16]:
to_be_dataset = CharDataset("To be or not to be", window_length=10)
for x, y in to_be_dataset:
    print(f"x={x}, y={y}")
    print(f"    decoded: x={decode_text(x)!r}, y={decode_text(y)!r}")

x=tensor([32, 27,  1, 14, 17,  1, 27, 30,  1, 26]), y=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27])
    decoded: x='to be or n', y='o be or no'
x=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27]), y=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32])
    decoded: x='o be or no', y=' be or not'
x=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32]), y=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1])
    decoded: x=' be or not', y='be or not '
x=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1]), y=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32])
    decoded: x='be or not ', y='e or not t'
x=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32]), y=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27])
    decoded: x='e or not t', y=' or not to'
x=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27]), y=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1])
    decoded: x=' or not to', y='or not to '
x=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1]), y=tensor([30,  1, 26, 27, 32,  1, 32, 27,  1, 14])
    decoded: x=

In [17]:
window_length = 50
batch_size = 512  # reduce if your GPU cannot handle such a large batch size
train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [18]:
import torch.nn as nn

torch.manual_seed(42)
embed = nn.Embedding(5, 3)  # 5 categories × 3D embeddings
embed(torch.tensor([[3, 2], [0, 2]]))

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

# Building and Training the Char-RNN Model

In [19]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [20]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)

In [21]:
n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

history = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
                n_epochs)

Epoch 1/20,                      train loss: 1.6040, train metric: 51.28%, valid metric: 51.98%
Epoch 2/20,                      train loss: 1.3843, train metric: 56.72%, valid metric: 52.83%
Epoch 3/20,                      train loss: 1.3547, train metric: 57.46%, valid metric: 53.64%
Epoch 4/20,                      train loss: 1.3403, train metric: 57.81%, valid metric: 53.45%
Epoch 5/20,                      train loss: 1.3320, train metric: 58.02%, valid metric: 53.32%
Epoch 6/20,                      train loss: 1.3264, train metric: 58.16%, valid metric: 53.79%
Epoch 7/20,                      train loss: 1.3225, train metric: 58.26%, valid metric: 53.71%
Epoch 8/20,                      train loss: 1.3193, train metric: 58.34%, valid metric: 54.09%
Epoch 9/20,                      train loss: 1.3167, train metric: 58.40%, valid metric: 54.33%
Epoch 10/20,                      train loss: 1.3148, train metric: 58.45%, valid metric: 54.08%
Epoch 11/20,                      train

In [22]:
torch.save(model.state_dict(), "my_shakespeare_model.pt")

In [23]:
model.eval()  # don't forget to switch the model to evaluation mode!
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = itos[predicted_char_id]  # correctly predicts "e"

In [24]:
predicted_char

'e'

# Generating Shakespearean Text

In [25]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])  # probas = 50%, 40%, and 10%
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [26]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return itos[predicted_char_id]

In [27]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [28]:
print(extend_text(model, "To be or not to b", temperature=0.01))

To be or not to be the county and the state
that thou hast should not stand the county and the co


In [29]:
print(extend_text(model, "To be or not to b", temperature=0.4))

To be or not to be so fare
the people that i have respected thee them first,
that he had the matt


In [30]:
print(extend_text(model, "To be or not to b", temperature=100))

To be or not to bmhf:my:r,k;s-h cqvvnfnfsut&-oq'ryoeen?x-hp:d,y&wv f3,dzrdzj-pilv?xpzc,fborp;'?$u


# Stateful RNNs

Until now, we have only used stateless RNNs: at each training iteration the model starts with a hidden state full of zeros, then it updates this state at each time step, and after the last time step, it throws the state away as it is not needed anymore. What if we instructed the RNN to preserve this final state after processing a training batch and use it as the initial state for the next training batch? This way the model could learn long-term patterns despite only backpropagating through short sequences. This is called a stateful RNN. Let's go over how to build one.

The model itself requires very little change: we only need to add a new hidden_states attribute, initialized to None, then save the hidden states after each batch is processed, and use them as the initial hidden states for the next batch. Note that we must call detach() on these states to ensure we don't backpropagate over this training iteration's computation graph at the next iteration (this would cause an error)."

In [31]:
class StatefulShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)
        self.hidden_states = None

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, hidden_states = self.gru(embeddings, self.hidden_states)
        self.hidden_states = hidden_states.detach()
        return self.output(outputs).permute(0, 2, 1)

In [32]:
# The main difficulty with stateful RNNs is preparing the dataset.
# Indeed, a stateful RNN only makes sense if each input sequence in a batch starts exactly where the corresponding sequence in the previous batch left off.
# To be more precise, the nth window in batch k must start exactly where the nth window in batch k – 1 stopped.
# For example, suppose the full encoded text is [1, 2, 3, .., 59, 60, 61] and you want to use a window length of 4, and a batch size of 5. The dataset could contain 3 batches like these, in this order:

# Batch #1:
# X=[[1,2,3,4], [13,14,15,16], [25,26,27,28], [37,38,39,40], [49,50,51,52]]
# Y=[[2,3,4,5], [14,15,16,17], [26,27,28,29], [38,39,40,41], [50,51,52,53]]

# Batch #2:
# X=[[5,6,7,8], [17,18,19,20], [29,30,31,32], [41,42,43,44], [53,54,55,56]]
# y=[[6,7,8,9], [18,19,20,21], [30,31,32,33], [42,43,44,45], [54,55,56,57]]

# Batch #3:
# X=[[9,10,11,12], [21,22,23,24], [33,34,35,36], [45,46,47,48], [57,58,59,60]]
# y=[[10,11,12,13], [22,23,24,25], [34,35,36,37], [46,47,48,49], [58,59,60,61]]

In [33]:
from torch.utils.data import Dataset, DataLoader

class StatefulCharDataset(Dataset):
    def __init__(self, text, window_length, batch_size):
        self.encoded_text = encode_text(text)
        self.window_length = window_length
        self.batch_size = batch_size
        n_consecutive_windows = (len(self.encoded_text) - 1) // window_length
        n_windows_per_slot = n_consecutive_windows // batch_size
        self.length = n_windows_per_slot * batch_size
        self.spacing = n_windows_per_slot * window_length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        start = ((idx % self.batch_size) * self.spacing
                 +(idx // self.batch_size) * self.window_length)
        end = start + self.window_length
        window = self.encoded_text[start : end]
        target = self.encoded_text[start + 1 : end + 1]
        return window, target

In [34]:
batch_size = 128
stateful_train_set = StatefulCharDataset(shakespeare_text[:1_000_000],
                                         window_length, batch_size)
stateful_train_loader = DataLoader(stateful_train_set, batch_size=batch_size,
                                   drop_last=True)
stateful_valid_set = StatefulCharDataset(shakespeare_text[1_000_000:1_060_000],
                                         window_length, batch_size)
stateful_valid_loader = DataLoader(stateful_valid_set, batch_size=batch_size,
                                   drop_last=True)
stateful_test_set = StatefulCharDataset(shakespeare_text[1_060_000:],
                                        window_length, batch_size)
stateful_test_loader = DataLoader(stateful_test_set, batch_size=batch_size,
                                  drop_last=True)

In [35]:
torch.manual_seed(42)

stateful_model = StatefulShakespeareModel(len(vocab)).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(stateful_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

def reset_hidden_states(model, epoch):
    model.hidden_states = None

history = train(stateful_model, optimizer, xentropy, accuracy, stateful_train_loader,
                stateful_valid_loader, n_epochs, epoch_callback=reset_hidden_states)

Epoch 1/10,                      train loss: 2.4742, train metric: 29.84%, valid metric: 39.29%
Epoch 2/10,                      train loss: 1.8777, train metric: 44.45%, valid metric: 44.55%
Epoch 3/10,                      train loss: 1.6904, train metric: 49.31%, valid metric: 48.19%
Epoch 4/10,                      train loss: 1.5955, train metric: 51.80%, valid metric: 50.18%
Epoch 5/10,                      train loss: 1.5377, train metric: 53.22%, valid metric: 51.35%
Epoch 6/10,                      train loss: 1.4994, train metric: 54.21%, valid metric: 52.09%
Epoch 7/10,                      train loss: 1.4714, train metric: 54.91%, valid metric: 52.81%
Epoch 8/10,                      train loss: 1.4504, train metric: 55.44%, valid metric: 53.33%
Epoch 9/10,                      train loss: 1.4331, train metric: 55.88%, valid metric: 53.62%
Epoch 10/10,                      train loss: 1.4183, train metric: 56.27%, valid metric: 54.03%


In [36]:
torch.save(stateful_model.state_dict(), "my_stateful_shakespeare_model.pt")

In [37]:
def extend_text_with_stateful_rnn(model, text, n_chars=80, temperature=1):
    model.hidden_states = None
    rnn_input = text
    for _ in range(n_chars):
        char = next_char(model, rnn_input, temperature)
        text += char
        rnn_input = char
    return text + "…"

In [38]:
torch.manual_seed(42)
stateful_model.eval()
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b",
                                    temperature=0.1))

To be or not to be
and the strength the senate the senate the death.

king richard iii:
the sing …


In [39]:
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=0.4))

To be or not to be
to deserved the grace to the borning the blessing to
the state and the lady of…


In [40]:
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=1))

To be or not to be fout:
then had bency to thou're enough her entland,
to to-self? that horse't g…


In [41]:
Out.clear()  # clear Jupyter's `Out` variable which saves all the cell outputs
del_vars(["accuracy", "embed", "encoded", "encoded_text", "optimizer", "probs",
          "samples", "x", "y", "shakespeare_text", "stateful_test_loader",
          "stateful_train_loader", "Y_logits", "stateful_valid_loader",
          "test_loader", "train_loader", "valid_loader", "xentropy"])

# Sentiment Analysis

In [42]:
from datasets import load_dataset

imdb_dataset = load_dataset('imdb')
split = imdb_dataset['train'].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [43]:
imdb_train_set[1]['text']

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [44]:
imdb_train_set[1]['label']

1

In [45]:
imdb_train_set[16]['text']

"Lillian Hellman's play, adapted by Dashiell Hammett with help from Hellman, becomes a curious project to come out of gritty Warner Bros. Paul Lukas, reprising his Broadway role and winning the Best Actor Oscar, plays an anti-Nazi German underground leader fighting the Fascists, dragging his American wife and three children all over Europe before finding refuge in the States (via the Mexico border). They settle in Washington with the wife's wealthy mother and brother, though a boarder residing in the manor is immediately suspicious of the newcomers and spends an awful lot of time down at the German Embassy playing poker. It seems to take forever for this drama to find its focus, and when we realize what the heart of the material is (the wise, honest, direct refugees teaching the clueless, head-in-the-sand Americans how the world has suddenly changed), it seems a little patronizing--the viewer is quite literally put in the relatives' place, being lectured to. Lukas has several speeches 

In [46]:
imdb_train_set[16]['label']

0

In [47]:
import tokenizers

In [48]:
bpe_model = tokenizers.models.BPE(unk_token='<unk>')
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ['<pad>', '<unk>']
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                             special_tokens=special_tokens)
train_reviews = [review['text'].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [49]:
tokenizers.pre_tokenizers.Whitespace().pre_tokenize_str('Salam, Hesen!')

[('Salam', (0, 5)), (',', (5, 6)), ('Hesen', (7, 12)), ('!', (12, 13))]

In [50]:
some_review = "what an awesome movie! 😍"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [51]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [52]:
bpe_token_ids = bpe_encoding.ids
bpe_token_ids

[303, 139, 373, 149, 240, 211, 4, 1]

In [53]:
bpe_tokenizer.get_vocab()['what']

303

In [54]:
bpe_tokenizer.token_to_id('what')

303

In [55]:
bpe_tokenizer.id_to_token(305)

'ough'

In [56]:
bpe_tokenizer.decode(bpe_token_ids)

'what an aw es ome movie !'

In [57]:
bpe_encoding.offsets

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]

In [59]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [60]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token='<pad>')
bpe_tokenizer.enable_truncation(max_length=500)

In [61]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [62]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [63]:
length = attention_mask.sum(dim=-1)
length

tensor([281, 114, 285])

# BBPE Tokenization

In [64]:
tokenizers.pre_tokenizers.ByteLevel().pre_tokenize_str(some_review)

[('Ġwhat', (0, 4)),
 ('Ġan', (4, 7)),
 ('Ġawesome', (7, 15)),
 ('Ġmovie', (15, 21)),
 ('!', (21, 22)),
 ('ĠðŁĺį', (22, 24))]

In [65]:
bbpe_model = tokenizers.models.BPE(unk_token='<unk>')
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                              special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(train_reviews, bbpe_trainer)

In [66]:
bbpe_encoding = bbpe_tokenizer.encode(some_review)
bbpe_tokens = bbpe_encoding.tokens
bbpe_tokens

['Ġwhat',
 'Ġan',
 'Ġaw',
 'es',
 'ome',
 'Ġmovie',
 '!',
 'Ġ',
 '<unk>',
 'Ł',
 'ĺ',
 'į']

In [67]:
bbpe_token_ids = bbpe_encoding.ids
bbpe_token_ids

[354, 216, 561, 148, 244, 232, 2, 107, 1, 125, 119, 112]

In [68]:
bbpe_decoded = bbpe_tokenizer.decode(bbpe_token_ids)
bbpe_decoded

'Ġwhat Ġan Ġaw es ome Ġmovie ! Ġ Ł ĺ į'

In [69]:
bbpe_decoded.replace(" ", '').replace('Ġ', ' ').strip()

'what an awesome movie! Łĺį'

# WordPiece

In [70]:
wp_model = tokenizers.models.WordPiece(unk_token='<unk>')
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
wp_trainer = tokenizers.trainers.WordPieceTrainer(vocab_size=1000,
                                                   special_tokens=special_tokens)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)

In [71]:
wp_encoding = wp_tokenizer.encode(some_review)
wp_tokens = wp_encoding.tokens
wp_tokens

['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '<unk>']

In [72]:
wp_token_ids = wp_encoding.ids
wp_token_ids

[443, 312, 635, 257, 354, 331, 4, 1]

In [73]:
wp_decoded = wp_tokenizer.decode(wp_token_ids)
wp_decoded

'what an aw ##es ##ome movie !'

In [74]:
wp_decoded.replace(' ##', '').replace(' !', '!')

'what an awesome movie!'

# UniGram

In [75]:
unigram_model = tokenizers.models.Unigram()
unigram_tokenizer = tokenizers.Tokenizer(unigram_model)
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
unigram_trainer = tokenizers.trainers.UnigramTrainer(vocab_size=1000,
                                                     special_tokens=special_tokens,
                                                     unk_token='<unk>')
unigram_tokenizer.train_from_iterator(train_reviews, unigram_trainer)

In [76]:
unigram_encoding = unigram_tokenizer.encode(some_review)
unigram_tokens = unigram_encoding.tokens
unigram_tokens

['what', 'an', 'a', 'w', 'e', 'some', 'movie', '!', '😍']

In [77]:
unigram_token_ids = unigram_encoding.ids
unigram_token_ids

[79, 37, 4, 40, 6, 70, 46, 74, 1]

In [78]:
unigram_decoded = unigram_tokenizer.decode(unigram_token_ids)
unigram_decoded

'what an a w e some movie !'

# PreTrained Tokenizers

In [79]:
import transformers

In [80]:
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True,
                               max_length=500)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [81]:
gpt2_encoding.keys()

KeysView({'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159,

In [82]:
gpt2_encoding

{'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159, 29847, 1

In [83]:
gpt2_token_ids = gpt2_encoding['input_ids'][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [84]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

# BERT

In [85]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained('bert-base-uncased')
bert_encoding = bert_tokenizer(train_reviews[:3], truncation=True,
                               max_length=500, padding=True,
                               return_tensors = 'pt')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [86]:
bert_encoding['input_ids']

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [87]:
bert_encoding['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [88]:
bert_encoding = bert_tokenizer(train_reviews[:3], truncation=True,
                               max_length=500, padding=True,
                               return_tensors = 'pt', add_special_tokens=False)

In [89]:
bert_encoding['input_ids']

tensor([[ 2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,  2411,
          2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,  2872,
          2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,  1000,
          1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,  1996,
          2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,  3861,
          3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,  1005,
          1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,  1012,
          1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,  3243,
          8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,  1012,
         19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,  2028,
          1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,  7987,
          1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,  1045,
          1005,  2310,  2196,  2464, 11209, 20206,  

# UniGram --> Albert, XLM-R, T5

In [90]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained('albert-base-v2')
albert_encoding = albert_tokenizer(train_reviews[:3], padding=True, truncation=True,
                                   max_length=500, add_special_tokens=False, return_tensors='pt')


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [91]:
albert_token_ids = albert_encoding['input_ids']
albert_token_ids

tensor([[  876,  5004,    18,   478,    57,    21,   394,  4173,     9,    59,
           478,   340,    70,   699,   101,    21,   171,  3336,    23,  1659,
          1037,    27,    14,   876,    13,     5,  4289,    28,    13,     7,
          4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,    18,
          1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,  1956,
           179,    14,  2151,  1434,    61,    90,   683,  2404,     9,   174,
            15,    32,    22,    18,  2210,    20,   361,    35,    26,    98,
            32,    25,     9,    14,  5427,   128,   832, 22427,    17,  4479,
         24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,  1429,
            50, 12891,     9, 22427,    25, 10356,    28,   550,    15,    17,
         24604,  3049,    53,    16,    33,   310, 11285,    20,   510,   601,
             9,     1,  5145,    13,   118,     1,  5145,    13,   118,     1,
            49, 14586,    30,    31,    22,   195,  

In [92]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object=bpe_tokenizer)
hf_encodings = hf_tokenizer(train_reviews[:3], padding=True, truncation=True, max_length=500, return_tensors='pt')
hf_encodings['input_ids']

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [93]:
def collate_fn(batch, tokenizer=bert_tokenizer):
  reviews = [review['text'] for review in batch]
  labels = [[review['label']] for review in batch]
  reviews = tokenizer(reviews, padding=True, truncation=True,
            max_length=200, return_tensors='pt')
  labels = torch.tensor(labels, dtype=torch.float32)

  return reviews, labels

In [94]:
batch_size = 256

In [95]:
imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size, collate_fn=collate_fn)

In [96]:
class SentimentAnalysisModel(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
               pad_id=0, dropout=0.2):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim,
                              padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, encoding):
    embeddings = self.embed(encoding['input_ids'])
    _outputs, hidden_states = self.gru(embeddings)
    return self.output(hidden_states[-1])


In [97]:
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

In [99]:
sequences = torch.tensor([
    [1, 5, 0, 0],
    [3, 7, 2, 1]
])

packed = pack_padded_sequence(sequences, lengths=(2, 4),
                              enforce_sorted=False, batch_first=True)
packed

PackedSequence(data=tensor([3, 1, 7, 5, 2, 1]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))

In [100]:
padded, length = pad_packed_sequence(packed, batch_first=True)
padded, length

(tensor([[1, 5, 0, 0],
         [3, 7, 2, 1]]),
 tensor([2, 4]))

In [102]:
class SentimentAnalysisModelPackedSeq(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
               pad_id=0, dropout=0.2):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim,
                              padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, encoding):
    embeddings = self.embed(encoding['input_ids'])
    lengths = encoding['attention_mask'].sum(dim=1)
    packed = pack_padded_sequence(embeddings, lengths.cpu(),
                                  enforce_sorted=False, batch_first=True)
    _outputs, hidden_states = self.gru(packed)
    return self.output(hidden_states[-1])


In [105]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_packed_seq = SentimentAnalysisModelPackedSeq(vocab_size).to(device)

n_epochs=10
optimizer = torch.optim.NAdam(imdb_model_packed_seq.parameters(), lr=1e-3)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)

history = train(imdb_model_packed_seq, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 0.6745, train metric: 57.35%, valid metric: 63.64%
Epoch 2/10,                      train loss: 0.5738, train metric: 71.10%, valid metric: 62.48%
Epoch 3/10,                      train loss: 0.4937, train metric: 77.13%, valid metric: 74.16%
Epoch 4/10,                      train loss: 0.4200, train metric: 82.02%, valid metric: 81.68%
Epoch 5/10,                      train loss: 0.3598, train metric: 85.21%, valid metric: 82.94%
Epoch 6/10,                      train loss: 0.3062, train metric: 87.79%, valid metric: 82.96%
Epoch 7/10,                      train loss: 0.2604, train metric: 89.96%, valid metric: 76.78%
Epoch 8/10,                      train loss: 0.2179, train metric: 92.06%, valid metric: 84.86%
Epoch 9/10,                      train loss: 0.1859, train metric: 93.65%, valid metric: 84.56%
Epoch 10/10,                      train loss: 0.1454, train metric: 95.35%, valid metric: 84.52%


In [106]:
class SentimentAnalysisModelPackedBiDirectional(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
               pad_id=0, dropout=0.2):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim,
                              padding_idx=pad_id)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                      batch_first=True, dropout=dropout, bidirectional=True)
    self.output = nn.Linear(2 * hidden_dim, 1)

  def forward(self, encoding):
    embeddings = self.embed(encoding['input_ids'])
    lengths = encoding['attention_mask'].sum(dim=1)
    packed = pack_padded_sequence(embeddings, lengths.cpu(),
                                  enforce_sorted=False, batch_first=True)
    _outputs, hidden_states = self.gru(packed)
    n_dims = self.output.in_features
    top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
    return self.output(top_states)


In [108]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_packed_bidi = SentimentAnalysisModelPackedBiDirectional(vocab_size).to(device)

n_epochs=10
optimizer = torch.optim.NAdam(imdb_model_packed_bidi.parameters(), lr=1e-3)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)

history = train(imdb_model_packed_seq, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 0.6395, train metric: 64.17%, valid metric: 62.08%
Epoch 2/10,                      train loss: 0.6401, train metric: 63.68%, valid metric: 62.08%
Epoch 3/10,                      train loss: 0.6398, train metric: 63.70%, valid metric: 62.08%
Epoch 4/10,                      train loss: 0.6399, train metric: 63.76%, valid metric: 62.08%
Epoch 5/10,                      train loss: 0.6410, train metric: 63.60%, valid metric: 62.08%
Epoch 6/10,                      train loss: 0.6404, train metric: 63.80%, valid metric: 62.08%
Epoch 7/10,                      train loss: 0.6407, train metric: 63.53%, valid metric: 62.08%
Epoch 8/10,                      train loss: 0.6391, train metric: 63.67%, valid metric: 62.08%
Epoch 9/10,                      train loss: 0.6408, train metric: 63.84%, valid metric: 62.08%
Epoch 10/10,                      train loss: 0.6405, train metric: 63.71%, valid metric: 62.08%


In [111]:
Out.clear()
del_vars(['albert_token_ids', 'attention_mask', 'bpe_batch_ids', 'encoded_text',
          "lengths", 'optimizer', 'padded', 'probs', 'sequences', 'samples',
          'x', 'xentropy', 'y', 'Y_logits'])



# Reusing Pretriend Embeddings and Language Models

In [114]:
bert_model = transformers.AutoModel.from_pretrained('bert-base-uncased')
bert_model.embeddings.word_embeddings

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

In [115]:
class SentimentAnalysisModelPreEmbed(nn.Module):
  def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2):
    super().__init__()

    weights = pretrained_embeddings.weight.data
    self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
    embed_dim = weights.shape[-1]
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                      batch_first=True, dropout=dropout, bidirectional=True)
    self.output = nn.Linear(2 * hidden_dim, 1)

  def forward(self, encoding):
    embeddings = self.embed(encoding['input_ids'])
    lengths = encoding['attention_mask'].sum(dim=1)
    packed = pack_padded_sequence(embeddings, lengths.cpu(),
                                  enforce_sorted=False, batch_first=True)
    _outputs, hidden_states = self.gru(packed)
    n_dims = self.output.in_features
    top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
    return self.output(top_states)


In [116]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_packed_pre_embed = SentimentAnalysisModelPreEmbed(bert_model.embeddings.word_embeddings).to(device)

n_epochs=10
optimizer = torch.optim.NAdam(imdb_model_packed_pre_embed.parameters(), lr=1e-3)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)

history = train(imdb_model_packed_seq, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 0.6409, train metric: 63.93%, valid metric: 62.08%
Epoch 2/10,                      train loss: 0.6405, train metric: 63.68%, valid metric: 62.08%
Epoch 3/10,                      train loss: 0.6395, train metric: 63.78%, valid metric: 62.08%
Epoch 4/10,                      train loss: 0.6407, train metric: 63.70%, valid metric: 62.08%
Epoch 5/10,                      train loss: 0.6402, train metric: 63.52%, valid metric: 62.08%
Epoch 6/10,                      train loss: 0.6401, train metric: 63.71%, valid metric: 62.08%
Epoch 7/10,                      train loss: 0.6404, train metric: 63.58%, valid metric: 62.08%
Epoch 8/10,                      train loss: 0.6403, train metric: 63.63%, valid metric: 62.08%
Epoch 9/10,                      train loss: 0.6406, train metric: 63.68%, valid metric: 62.08%
Epoch 10/10,                      train loss: 0.6396, train metric: 63.66%, valid metric: 62.08%


In [117]:
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, truncation=True, max_lenhgth=200, return_tensors='pt')

bert_output = bert_model(**bert_encoding)
bert_output

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2470, -0.4977,  0.1271,  ...,  0.0853,  0.5811,  0.4216],
         [ 0.0992,  0.4679, -0.2322,  ..., -0.0807,  0.8464, -0.6777],
         [ 0.9692,  0.2901, -0.1934,  ..., -0.1843,  0.0797,  0.0119],
         ...,
         [-0.1233, -0.1043,  0.5893,  ...,  0.5469,  0.5141, -0.2301],
         [ 0.0230, -0.0585,  0.3483,  ..., -0.0144,  0.3055,  0.3365],
         [ 0.1094,  0.0173,  0.3051,  ..., -0.1052,  0.2107,  0.1548]],

        [[-0.0101, -0.3241,  0.3658,  ..., -0.4497,  0.5121, -0.1260],
         [-0.2964, -0.2872, -0.5425,  ...,  0.6877,  1.3121, -1.5069],
         [-0.4449, -0.8080, -0.4515,  ...,  0.5536,  0.8961, -0.3652],
         ...,
         [ 0.0344, -0.4247,  0.5519,  ...,  0.4544,  0.0646, -0.5471],
         [ 0.3546, -0.2916,  0.4812,  ..., -0.0657,  0.1345, -0.0964],
         [ 0.2253, -0.4641,  0.6720,  ...,  0.1906,  0.1024, -0.3169]],

        [[ 0.3226, -0.8728,  0.3269,  ...,  0.0659,  

In [118]:
bert_output.last_hidden_state.shape

torch.Size([3, 234, 768])

In [119]:
bert_output.pooler_output.shape

torch.Size([3, 768])

In [120]:
del_vars(['bert_model'])

In [123]:
class SentimentAnalysisBertModel(nn.Module):
  def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
    super().__init__()

    self.bert = transformers.AutoModel.from_pretrained('bert-base-uncased')
    embed_dim = self.bert.config.hidden_size
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                      batch_first=True, dropout=dropout)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, encoding):
    contextualized_embeddings = self.bert(**encoding).last_hidden_state
    lengths = encoding['attention_mask'].sum(dim=1)
    packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(),
                                  enforce_sorted=False, batch_first=True)
    _outputs, hidden_states = self.gru(packed)
    return self.output(hidden_states[-1])

In [124]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisBertModel().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs=5
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters(), lr=1e-3)
accuracy = torchmetrics.Accuracy(task='binary').to(device)

history = train(imdb_model_bert, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5,                      train loss: 0.5210, train metric: 73.33%, valid metric: 83.32%
Epoch 2/5,                      train loss: 0.3408, train metric: 85.26%, valid metric: 87.74%
Epoch 3/5,                      train loss: 0.2944, train metric: 87.78%, valid metric: 88.12%
Epoch 4/5,                      train loss: 0.2769, train metric: 88.57%, valid metric: 85.50%
Epoch 5/5,                      train loss: 0.2582, train metric: 89.28%, valid metric: 89.18%


In [125]:
del_vars(['imdb_model_bert'])

In [126]:
class SentimentAnalysisModelBert2(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.last_hidden_state[:, 0])

In [127]:
class SentimentAnalysisModelBert3(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.pooler_output)

In [ ]:
torch.manual_seed(42)

imdb_model_bert3 = SentimentAnalysisModelBert3().to(device)
imdb_model_bert3.bert.requires_grad_(False)

n_epochs = 5
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert3.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batch 41/79, loss=0.6770, train_metric=58.09%

In [ ]:
imdb_model_bert3.bert.pooler.requires_grad_(True)

history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

In [ ]:
del_vars(["imdb_model_bert3"])